<a href="https://colab.research.google.com/github/NABOTH-14/Banking-Analytics/blob/main/Banking_Analytics_data_preparation_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Synthetic Banking Analytics dataset generator.
For a Data Analyst portfolio project (Excel, SQL, Power BI).
All data is fictional. No real customer, account or personal information is used.
"""

import numpy as np
import pandas as pd
from datetime import date, timedelta

RNG = np.random.default_rng(20260101)

# ----------------------------------------------------------------------
# 1. BRANCHES  (10 branches)
# ----------------------------------------------------------------------
branches = pd.DataFrame([
    ("B001", "Lusaka Main",   "Lusaka",       "Lusaka"),
    ("B002", "Manda Hill",    "Lusaka",       "Lusaka"),
    ("B003", "Cairo Road",    "Lusaka",       "Lusaka"),
    ("B004", "Woodlands",     "Lusaka",       "Lusaka"),
    ("B005", "Kitwe",         "Copperbelt",   "Kitwe"),
    ("B006", "Ndola",         "Copperbelt",   "Ndola"),
    ("B007", "Livingstone",   "Southern",     "Livingstone"),
    ("B008", "Kabwe",         "Central",      "Kabwe"),
    ("B009", "Mongu",         "Western",      "Mongu"),
    ("B010", "Chipata",       "Eastern",      "Chipata"),
], columns=["Branch_ID", "Branch_Name", "Province", "City"])

# ----------------------------------------------------------------------
# 2. CUSTOMERS  (1,000)
# ----------------------------------------------------------------------
N_CUST = 1000

# Province -> list of cities, with a realistic weight per province
province_cities = {
    "Lusaka":       ["Lusaka"],
    "Copperbelt":   ["Kitwe", "Ndola"],
    "Southern":     ["Livingstone"],
    "Central":      ["Kabwe"],
    "Eastern":      ["Chipata"],
    "Western":      ["Mongu"],
    "Northern":     ["Kasama"],
    "Luapula":      ["Mansa"],
    "Muchinga":     ["Chinsali"],
    "North-Western":["Solwezi"],
}

# Realistic population/economic weighting (Lusaka & Copperbelt dominate)
province_weights = np.array([0.30, 0.22, 0.08, 0.07, 0.06, 0.05, 0.05, 0.04, 0.04, 0.09])
province_weights = province_weights / province_weights.sum()
provinces = list(province_cities.keys())

# Branch lookup by province (Lusaka has 4 branches, others 1)
branch_by_province = {
    "Lusaka":       ["B001", "B002", "B003", "B004"],
    "Copperbelt":   ["B005", "B006"],
    "Southern":     ["B007"],
    "Central":      ["B008"],
    "Western":      ["B009"],
    "Eastern":      ["B010"],
    "Northern":     ["B005"],   # nearest branch (Kitwe) - realistic rural routing
    "Luapula":      ["B005"],
    "Muchinga":     ["B010"],
    "North-Western":["B005"],
}

segments = ["Student", "Individual", "SME", "Corporate"]
segment_weights = [0.18, 0.62, 0.15, 0.05]

genders = ["Male", "Female"]

def random_join_date():
    # Join dates spread over ~8 years, weighted slightly toward recent years
    days_back = int(RNG.gamma(2.0, 400))
    days_back = min(days_back, 8 * 365)
    return date(2026, 1, 1) - timedelta(days=days_back)

cust_rows = []
for i in range(1, N_CUST + 1):
    cid = f"C{i:05d}"
    prov = RNG.choice(provinces, p=province_weights)
    city = RNG.choice(province_cities[prov])
    branch = RNG.choice(branch_by_province[prov])
    seg = RNG.choice(segments, p=segment_weights)

    if seg == "Student":
        age = int(np.clip(RNG.normal(22, 3), 17, 30))
    elif seg == "Individual":
        age = int(np.clip(RNG.normal(38, 11), 18, 75))
    elif seg == "SME":
        age = int(np.clip(RNG.normal(42, 9), 25, 70))
    else:  # Corporate
        age = int(np.clip(RNG.normal(45, 8), 28, 70))

    gender = RNG.choice(genders)
    cust_rows.append([cid, age, gender, prov, city, branch, seg, random_join_date()])

customers = pd.DataFrame(cust_rows, columns=[
    "Customer_ID", "Age", "Gender", "Province", "City",
    "Branch_ID", "Customer_Segment", "Join_Date"
])

# ----------------------------------------------------------------------
# 3. ACCOUNTS  (~1,000, one per customer mostly)
# ----------------------------------------------------------------------
acc_types = ["Savings", "Current", "Business", "Student"]
# Student segment -> Student account; Corporate -> Business; etc.
def pick_account_type(seg):
    if seg == "Student":
        return RNG.choice(["Student", "Savings"], p=[0.8, 0.2])
    if seg == "Corporate":
        return RNG.choice(["Business", "Current"], p=[0.7, 0.3])
    if seg == "SME":
        return RNG.choice(["Business", "Current", "Savings"], p=[0.5, 0.3, 0.2])
    return RNG.choice(["Savings", "Current"], p=[0.75, 0.25])

statuses = ["Active", "Inactive", "Dormant", "Closed"]
status_weights = [0.72, 0.14, 0.09, 0.05]

def opening_balance(seg, atype):
    if seg == "Corporate":
        return round(float(RNG.lognormal(11.5, 1.1)), 2)     # ~ tens of thousands to millions
    if seg == "SME":
        return round(float(RNG.lognormal(10.2, 1.0)), 2)
    if seg == "Student":
        return round(float(RNG.lognormal(6.3, 0.8)), 2)      # small balances
    return round(float(RNG.lognormal(8.6, 1.1)), 2)

acc_rows = []
for _, c in customers.iterrows():
    atype = pick_account_type(c["Customer_Segment"])
    status = RNG.choice(statuses, p=status_weights)
    bal = opening_balance(c["Customer_Segment"], atype)
    # Account opened on/after join date
    open_date = c["Join_Date"] + timedelta(days=int(RNG.integers(0, 60)))
    acc_rows.append([
        f"A{len(acc_rows)+1:05d}",
        c["Customer_ID"],
        atype,
        status,
        bal,
        open_date,
    ])

# Add ~30 extra accounts for a few customers (multi-product holders)
extra_customers = RNG.choice(customers["Customer_ID"], size=30, replace=False)
for cid in extra_customers:
    c = customers[customers["Customer_ID"] == cid].iloc[0]
    atype = pick_account_type(c["Customer_Segment"])
    status = RNG.choice(statuses, p=status_weights)
    bal = opening_balance(c["Customer_Segment"], atype)
    open_date = c["Join_Date"] + timedelta(days=int(RNG.integers(0, 400)))
    acc_rows.append([
        f"A{len(acc_rows)+1:05d}",
        cid, atype, status, bal, open_date
    ])

accounts = pd.DataFrame(acc_rows, columns=[
    "Account_ID", "Customer_ID", "Account_Type", "Account_Status",
    "Opening_Balance", "Account_Open_Date"
])

# ----------------------------------------------------------------------
# 4. TRANSACTIONS  (~6,000, 2026-01-01 to 2026-08-31)
# ----------------------------------------------------------------------
start = date(2026, 1, 1)
end = date(2026, 8, 31)
total_days = (end - start).days + 1

tx_types = ["Deposit", "Withdrawal", "Transfer", "Payment", "Loan Repayment"]
tx_type_weights = [0.30, 0.28, 0.18, 0.16, 0.08]

channels = ["Mobile Banking", "ATM", "Branch", "Internet Banking", "POS"]
channel_weights = [0.42, 0.18, 0.15, 0.13, 0.12]

tx_status = ["Completed", "Failed", "Reversed"]
tx_status_weights = [0.92, 0.055, 0.025]

# Customer activity profile: some very active, some quiet
cust_ids = customers["Customer_ID"].tolist()
# Assign each customer an activity weight
activity = RNG.gamma(1.4, 1.0, size=len(cust_ids))
activity = activity / activity.sum()

# Choose 6,000 customer draws weighted by activity (some customers get 0)
n_tx = 6000
chosen_idx = RNG.choice(len(cust_ids), size=n_tx, p=activity)
# Force a handful of customers to have zero transactions
zero_activity = set(RNG.choice(cust_ids, size=25, replace=False))

# Transaction amount depends on segment
seg_amount_params = {
    "Student":     (6.5, 0.9),   # lognormal mu, sigma -> small
    "Individual":  (8.2, 1.0),
    "SME":         (9.6, 1.1),
    "Corporate":   (11.0, 1.2),
}

def amount_for(seg, ttype):
    mu, sigma = seg_amount_params[seg]
    if ttype == "Loan Repayment":
        mu += 0.6
    if ttype == "Deposit":
        mu += 0.2
    return round(float(RNG.lognormal(mu, sigma)), 2)

# Monthly seasonality (Jan..Aug 2026)
month_weights = np.array([0.09, 0.11, 0.13, 0.12, 0.13, 0.13, 0.14, 0.15])
month_weights = month_weights / month_weights.sum()

# Build day weights across the range with month seasonality
day_weights = np.zeros(total_days)
for d in range(total_days):
    cur = start + timedelta(days=d)
    day_weights[d] = month_weights[cur.month - 1]
day_weights = day_weights / day_weights.sum()

cust_lookup = customers.set_index("Customer_ID")

tx_rows = []
tx_id = 1
for _ in range(n_tx):
    idx = RNG.choice(len(cust_ids))  # already weighted above via chosen_idx, but re-draw for day
    # Use chosen_idx properly:
    pass

# Re-do cleanly:
chosen_idx = RNG.choice(len(cust_ids), size=n_tx, p=activity)

for k in range(n_tx):
    cid = cust_ids[chosen_idx[k]]
    if cid in zero_activity:
        continue
    c = cust_lookup.loc[cid]
    seg = c["Customer_Segment"]
    day = int(RNG.choice(total_days, p=day_weights))
    tx_date = start + timedelta(days=day)
    ttype = RNG.choice(tx_types, p=tx_type_weights)
    channel = RNG.choice(channels, p=channel_weights)
    # Branch: mostly the customer's home branch; sometimes another
    if RNG.random() < 0.8:
        branch = c["Branch_ID"]
    else:
        branch = RNG.choice(branches["Branch_ID"])
    amt = amount_for(seg, ttype)
    status = RNG.choice(tx_status, p=tx_status_weights)
    tx_rows.append([
        f"T{tx_id:06d}", cid, tx_date, ttype, channel, branch,
        amt, status
    ])
    tx_id += 1

transactions = pd.DataFrame(tx_rows, columns=[
    "Transaction_ID", "Customer_ID", "Transaction_Date", "Transaction_Type",
    "Channel", "Branch_ID", "Amount", "Transaction_Status"
])

# ----------------------------------------------------------------------
# 5. LOANS  (~450)
# ----------------------------------------------------------------------
loan_types = ["Personal", "SME", "Salary Advance", "Agricultural"]
loan_type_weights = [0.45, 0.20, 0.25, 0.10]

loan_status = ["Active", "Paid", "Defaulted", "Rejected"]
loan_status_weights = [0.50, 0.32, 0.08, 0.10]

terms = [3, 6, 12, 24, 36, 48]
term_weights = [0.10, 0.18, 0.30, 0.22, 0.12, 0.08]

loan_amount_params = {
    "Personal":       (10.0, 0.8),
    "SME":            (11.3, 0.9),
    "Salary Advance": (9.6, 0.5),
    "Agricultural":   (10.7, 0.9),
}
loan_rate = {
    "Personal":       (0.22, 0.04),
    "SME":            (0.20, 0.04),
    "Salary Advance": (0.15, 0.03),
    "Agricultural":   (0.18, 0.03),
}

n_loans = 450
# Eligible customers: not Student (mostly), and not the zero-activity ones
eligible = customers[customers["Customer_Segment"] != "Student"]["Customer_ID"].tolist()
loan_cust = RNG.choice(eligible, size=n_loans, replace=True)

loan_rows = []
for i in range(n_loans):
    cid = loan_cust[i]
    c = cust_lookup.loc[cid]
    ltype = RNG.choice(loan_types, p=loan_type_weights)
    mu, sigma = loan_amount_params[ltype]
    amt = round(float(RNG.lognormal(mu, sigma)), 2)
    # Cap at realistic Zambian ceiling
    amt = min(amt, 5_000_000.0)
    status = RNG.choice(loan_status, p=loan_status_weights)
    # Application date within last 2 years
    days_back = int(RNG.integers(30, 730))
    app_date = end - timedelta(days=days_back)
    r_mu, r_sd = loan_rate[ltype]
    rate = round(float(np.clip(RNG.normal(r_mu, r_sd), 0.08, 0.35)), 4)
    term = int(RNG.choice(terms, p=term_weights))
    loan_rows.append([
        f"L{i+1:05d}", cid, ltype, amt, status, app_date, rate, term
    ])

loans = pd.DataFrame(loan_rows, columns=[
    "Loan_ID", "Customer_ID", "Loan_Type", "Loan_Amount",
    "Loan_Status", "Application_Date", "Interest_Rate", "Loan_Term_Months"
])

# ----------------------------------------------------------------------
# 6. INTENTIONAL DATA-QUALITY ISSUES
# ----------------------------------------------------------------------
# 6a. Missing Gender (a few)
customers.loc[customers.sample(8, random_state=1).index, "Gender"] = np.nan

# 6b. Inconsistent channel values (case + spacing)
tx_idx = transactions.sample(frac=1.0, random_state=2).index
for i, idx in enumerate(tx_idx[:25]):
    v = transactions.at[idx, "Channel"]
    if i % 3 == 0:
        transactions.at[idx, "Channel"] = v.lower()
    elif i % 3 == 1:
        transactions.at[idx, "Channel"] = v.upper()
    else:
        transactions.at[idx, "Channel"] = f"  {v}  "

# 6c. Inconsistent province / city spellings
bad_prov = {
    "Lusaka": ["lusaka", "LUSAKA", "Lusaka "],
    "Copperbelt": ["Copper Belt", "copperbelt", "Copperbelt "],
    "Southern": ["southern", "Southern "],
}
for true_prov, variants in bad_prov.items():
    idxs = customers[customers["Province"] == true_prov].sample(3, random_state=3).index
    for j, idx in enumerate(idxs):
        customers.at[idx, "Province"] = variants[j % len(variants)]

# 6d. Leading/trailing spaces in some text fields
for col in ["Customer_Segment", "Account_Type", "Loan_Type"]:
    if col in customers.columns:
        idxs = customers.sample(4, random_state=4).index
        for idx in idxs:
            customers.at[idx, col] = f" {customers.at[idx, col]} "
    if col in accounts.columns:
        idxs = accounts.sample(4, random_state=5).index
        for idx in idxs:
            accounts.at[idx, col] = f" {accounts.at[idx, col]} "
    if col in loans.columns:
        idxs = loans.sample(4, random_state=6).index
        for idx in idxs:
            loans.at[idx, col] = f" {loans.at[idx, col]} "

# 6e. Missing transaction Amounts
missing_amt_idx = transactions.sample(20, random_state=7).index
transactions.loc[missing_amt_idx, "Amount"] = np.nan

# 6f. Inconsistent date formats (store as strings for a few rows)
# We'll write the dates normally, but a handful will be strings in non-ISO format.
# Easier: introduce a separate "date as text" column? No — instead, we convert
# a few date cells to strings like "31/08/2026" or "Aug 31, 2026".
mixed_date_idx = transactions.sample(10, random_state=8).index
transactions["Transaction_Date"] = transactions["Transaction_Date"].astype(object)
for idx in mixed_date_idx:
    d = transactions.at[idx, "Transaction_Date"]
    if isinstance(d, date):
        transactions.at[idx, "Transaction_Date"] = d.strftime("%d/%m/%Y")

# 6g. Duplicate transaction records (exact duplicates)
dupes = transactions.sample(15, random_state=9)
transactions = pd.concat([transactions, dupes], ignore_index=True)

# 6h. Invalid / unusual transaction amounts
# Negative amounts (should not happen for these types) and extreme values
invalid_idx = transactions.sample(6, random_state=10).index
for i, idx in enumerate(invalid_idx):
    if i % 2 == 0:
        transactions.at[idx, "Amount"] = -abs(transactions.at[idx, "Amount"])
    else:
        transactions.at[idx, "Amount"] = round(float(RNG.uniform(500_000, 2_000_000)), 2)

# 6i. A few customers with no transactions already handled via zero_activity.

# 6j. Inactive accounts with recent transactions
# Pick 10 inactive/dormant accounts and inject a transaction for them in Aug 2026
inactive_accs = accounts[accounts["Account_Status"].isin(["Inactive", "Dormant"])]
inactive_custs = inactive_accs["Customer_ID"].sample(10, random_state=11).tolist()
extra_tx = []
for j, cid in enumerate(inactive_custs):
    c = cust_lookup.loc[cid]
    extra_tx.append([
        f"T{90000+j:06d}", cid,
        date(2026, 8, int(RNG.integers(1, 29))),
        RNG.choice(tx_types, p=tx_type_weights),
        RNG.choice(channels, p=channel_weights),
        c["Branch_ID"],
        round(float(RNG.lognormal(8.5, 1.0)), 2),
        "Completed",
    ])
transactions = pd.concat(
    [transactions, pd.DataFrame(extra_tx, columns=transactions.columns)],
    ignore_index=True
)

# 6k. A few broken foreign keys (customer not in Customers)
bad_fk_idx = transactions.sample(4, random_state=12).index
for idx in bad_fk_idx:
    transactions.at[idx, "Customer_ID"] = "C99999"

# Shuffle transactions so duplicates/errors aren't clustered
transactions = transactions.sample(frac=1.0, random_state=13).reset_index(drop=True)

# ----------------------------------------------------------------------
# 7. DATA QUALITY NOTES SHEET
# ----------------------------------------------------------------------
notes = pd.DataFrame({
    "Issue_Category": [
        "Missing values",
        "Missing values",
        "Inconsistent text casing",
        "Inconsistent text casing",
        "Inconsistent province/city spelling",
        "Leading/trailing whitespace",
        "Duplicate records",
        "Inconsistent date formats",
        "Invalid or unusual amounts",
        "Customers with no transactions",
        "Status/activity inconsistency",
        "Broken foreign keys",
    ],
    "Description": [
        "A small number of Gender values are blank in the Customers sheet.",
        "A small number of Amount values are blank in the Transactions sheet.",
        "Some Channel values appear as 'Mobile Banking', 'mobile banking', 'MOBILE BANKING', etc.",
        "Some text fields (segments, account types, loan types) have inconsistent casing or extra spaces.",
        "A few Province values use variants such as 'lusaka', 'LUSAKA', 'Copper Belt'.",
        "A few text fields contain leading/trailing spaces.",
        "A small number of transaction rows are exact duplicates.",
        "A few Transaction_Date values are stored as text in non-ISO formats (e.g. DD/MM/YYYY).",
        "A few transaction amounts are negative or unusually large and should be investigated.",
        "A small number of customers have no transactions at all.",
        "A few Inactive/Dormant accounts have recent (Aug 2026) transactions.",
        "A few transaction Customer_ID values do not exist in the Customers sheet.",
    ],
    "Suggested_Cleaning_Action": [
        "Decide whether to impute, flag or exclude depending on analysis.",
        "Investigate source system; consider excluding from amount-based aggregates.",
        "Standardise to a single canonical value (e.g. 'Mobile Banking').",
        "TRIM and PROPER / UPPER as appropriate.",
        "Map variants to the official province name.",
        "Apply TRIM (Excel) or strip() (Python/SQL).",
        "De-duplicate on all columns or on Transaction_ID.",
        "Parse to a consistent date type; investigate why formats differ.",
        "Flag and exclude from totals; investigate possible data-entry errors.",
        "Decide whether to include in 'active customer' definitions.",
        "Flag as data-quality exceptions; investigate account status logic.",
        "Flag as referential-integrity issues; exclude from customer-level joins.",
    ],
})

# ----------------------------------------------------------------------
# 8. WRITE EXCEL WORKBOOK
# ----------------------------------------------------------------------
out_path = "Zanaco_Banking_Analytics_Raw_Data.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    customers.to_excel(writer, sheet_name="Customers", index=False)
    accounts.to_excel(writer, sheet_name="Accounts", index=False)
    transactions.to_excel(writer, sheet_name="Transactions", index=False)
    loans.to_excel(writer, sheet_name="Loans", index=False)
    branches.to_excel(writer, sheet_name="Branches", index=False)
    notes.to_excel(writer, sheet_name="DATA_QUALITY_NOTES", index=False)

print(f"Workbook written to: {out_path}")
print("Sheets: Customers, Accounts, Transactions, Loans, Branches, DATA_QUALITY_NOTES")

Workbook written to: Zanaco_Banking_Analytics_Raw_Data.xlsx
Sheets: Customers, Accounts, Transactions, Loans, Branches, DATA_QUALITY_NOTES


In [ ]:
print ('hello world')

hello world


In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# ZANACO BANKING ANALYTICS PROJECT
# Synthetic Banking Dataset Generator
# ============================================================

np.random.seed(42)

# ------------------------------------------------------------
# 1. BRANCH DATA
# ------------------------------------------------------------

branches = pd.DataFrame({
    "Branch_ID": [
        "B001", "B002", "B003", "B004", "B005",
        "B006", "B007", "B008", "B009", "B010"
    ],
    "Branch_Name": [
        "Lusaka Main",
        "Manda Hill",
        "Cairo Road",
        "Woodlands",
        "Kitwe",
        "Ndola",
        "Livingstone",
        "Kabwe",
        "Mongu",
        "Chipata"
    ],
    "Province": [
        "Lusaka", "Lusaka", "Lusaka", "Lusaka",
        "Copperbelt", "Copperbelt", "Southern",
        "Central", "Western", "Eastern"
    ],
    "City": [
        "Lusaka", "Lusaka", "Lusaka", "Lusaka",
        "Kitwe", "Ndola", "Livingstone",
        "Kabwe", "Mongu", "Chipata"
    ]
})


# ------------------------------------------------------------
# 2. CUSTOMERS
# ------------------------------------------------------------

n_customers = 1000

customer_ids = [
    f"C{i:05d}" for i in range(1, n_customers + 1)
]

# Branch weights - larger branches receive more customers
branch_weights = [
    0.16, 0.12, 0.13, 0.09,
    0.11, 0.10, 0.07, 0.07,
    0.08, 0.07
]

customer_branch = np.random.choice(
    branches["Branch_ID"],
    size=n_customers,
    p=branch_weights
)

province_map = branches.set_index("Branch_ID")["Province"].to_dict()
city_map = branches.set_index("Branch_ID")["City"].to_dict()

customers = pd.DataFrame({
    "Customer_ID": customer_ids,

    "Age": np.random.randint(
        18, 66, size=n_customers
    ),

    "Gender": np.random.choice(
        ["Male", "Female"],
        size=n_customers,
        p=[0.48, 0.52]
    ),

    "Province": [
        province_map[x] for x in customer_branch
    ],

    "City": [
        city_map[x] for x in customer_branch
    ],

    "Branch_ID": customer_branch,

    "Customer_Segment": np.random.choice(
        ["Student", "Individual", "SME", "Corporate"],
        size=n_customers,
        p=[0.12, 0.58, 0.24, 0.06]
    ),

    "Join_Date": pd.to_datetime(
        np.random.choice(
            pd.date_range(
                "2023-01-01",
                "2026-06-30"
            ),
            size=n_customers
        )
    )
})


# ------------------------------------------------------------
# 3. ACCOUNTS
# ------------------------------------------------------------

account_types = np.random.choice(
    ["Savings", "Current", "Business", "Student"],
    size=n_customers,
    p=[0.52, 0.20, 0.20, 0.08]
)

accounts = pd.DataFrame({
    "Account_ID": [
        f"A{i:06d}"
        for i in range(1, n_customers + 1)
    ],

    "Customer_ID": customer_ids,

    "Account_Type": account_types,

    "Account_Status": np.random.choice(
        ["Active", "Inactive", "Dormant", "Closed"],
        size=n_customers,
        p=[0.70, 0.16, 0.08, 0.06]
    ),

    "Opening_Balance": np.round(
        np.random.lognormal(
            mean=7.5,
            sigma=1.0,
            size=n_customers
        ),
        2
    ),

    "Account_Open_Date": pd.to_datetime(
        np.random.choice(
            pd.date_range(
                "2023-01-01",
                "2026-06-30"
            ),
            size=n_customers
        )
    )
})


# ------------------------------------------------------------
# 4. TRANSACTIONS
# ------------------------------------------------------------

n_transactions = 6000

transaction_customers = np.random.choice(
    customer_ids,
    size=n_transactions
)

transaction_dates = pd.to_datetime(
    np.random.choice(
        pd.date_range(
            "2026-01-01",
            "2026-08-31"
        ),
        size=n_transactions
    )
)

transaction_types = np.random.choice(
    [
        "Deposit",
        "Withdrawal",
        "Transfer",
        "Payment",
        "Loan Repayment"
    ],
    size=n_transactions,
    p=[0.28, 0.20, 0.22, 0.22, 0.08]
)

channels = np.random.choice(
    [
        "Mobile Banking",
        "ATM",
        "Branch",
        "Internet Banking",
        "POS"
    ],
    size=n_transactions,
    p=[0.34, 0.25, 0.16, 0.15, 0.10]
)

transaction_status = np.random.choice(
    [
        "Completed",
        "Failed",
        "Reversed"
    ],
    size=n_transactions,
    p=[0.94, 0.04, 0.02]
)

# Create transaction amounts
amounts = np.random.lognormal(
    mean=6.7,
    sigma=1.0,
    size=n_transactions
)

# Keep amounts within realistic bounds
amounts = np.clip(
    amounts,
    20,
    50000
)

amounts = np.round(amounts, 2)

transaction_branches = np.random.choice(
    branches["Branch_ID"],
    size=n_transactions,
    p=branch_weights
)

transactions = pd.DataFrame({
    "Transaction_ID": [
        f"T{i:07d}"
        for i in range(1, n_transactions + 1)
    ],

    "Customer_ID": transaction_customers,

    "Transaction_Date": transaction_dates,

    "Transaction_Type": transaction_types,

    "Channel": channels,

    "Branch_ID": transaction_branches,

    "Amount": amounts,

    "Transaction_Status": transaction_status
})


# ------------------------------------------------------------
# 5. LOANS
# ------------------------------------------------------------

n_loans = 450

loan_customer_ids = np.random.choice(
    customer_ids,
    size=n_loans,
    replace=False
)

loans = pd.DataFrame({
    "Loan_ID": [
        f"L{i:05d}"
        for i in range(1, n_loans + 1)
    ],

    "Customer_ID": loan_customer_ids,

    "Loan_Type": np.random.choice(
        [
            "Personal",
            "SME",
            "Salary Advance",
            "Agricultural"
        ],
        size=n_loans,
        p=[0.45, 0.25, 0.20, 0.10]
    ),

    "Loan_Amount": np.round(
        np.clip(
            np.random.lognormal(
                mean=9.8,
                sigma=0.8,
                size=n_loans
            ),
            1000,
            250000
        ),
        2
    ),

    "Loan_Status": np.random.choice(
        [
            "Active",
            "Paid",
            "Defaulted",
            "Rejected"
        ],
        size=n_loans,
        p=[0.50, 0.35, 0.07, 0.08]
    ),

    "Application_Date": pd.to_datetime(
        np.random.choice(
            pd.date_range(
                "2025-01-01",
                "2026-08-31"
            ),
            size=n_loans
        )
    ),

    "Interest_Rate": np.round(
        np.random.uniform(
            10,
            35,
            size=n_loans
        ),
        2
    ),

    "Loan_Term_Months": np.random.choice(
        [3, 6, 12, 24, 36, 48],
        size=n_loans,
        p=[0.08, 0.15, 0.30, 0.25, 0.15, 0.07]
    )
})


# ============================================================
# 6. INTRODUCE DATA QUALITY PROBLEMS
# ============================================================

# Missing gender values
customers.loc[
    [12, 87, 341, 612],
    "Gender"
] = np.nan


# Missing transaction amounts
transactions.loc[
    [55, 912, 2310, 4780],
    "Amount"
] = np.nan


# Inconsistent channel formatting
transactions.loc[
    [101, 102],
    "Channel"
] = "mobile banking"

transactions.loc[
    [150],
    "Channel"
] = " MOBILE BANKING "

transactions.loc[
    [205],
    "Channel"
] = "ATM "


# Leading/trailing spaces in text
customers.loc[
    [45, 178],
    "City"
] = customers.loc[
    [45, 178],
    "City"
].apply(lambda x: " " + x)

accounts.loc[
    [72, 415],
    "Account_Type"
] = accounts.loc[
    [72, 415],
    "Account_Type"
].apply(lambda x: x + " ")


# Inconsistent province spelling
customers.loc[
    [230],
    "Province"
] = "Lusaka Province"


# Unusual transaction amounts
transactions.loc[
    300,
    "Amount"
] = 0

transactions.loc[
    301,
    "Amount"
] = -500


# ------------------------------------------------------------
# Duplicate transaction records
# ------------------------------------------------------------

duplicate_rows = transactions.iloc[
    [500, 1000, 1500]
].copy()

transactions = pd.concat(
    [transactions, duplicate_rows],
    ignore_index=True
)


# ============================================================
# 7. DATA QUALITY NOTES
# ============================================================

data_quality_notes = pd.DataFrame({
    "Issue_Type": [
        "Missing Values",
        "Missing Values",
        "Inconsistent Text",
        "Inconsistent Text",
        "Duplicate Records",
        "Invalid/Unusual Values",
        "Whitespace",
        "Inconsistent Geography"
    ],

    "Affected_Table": [
        "Customers",
        "Transactions",
        "Transactions",
        "Customers",
        "Transactions",
        "Transactions",
        "Customers/Accounts",
        "Customers"
    ],

    "Description": [
        "Some customer gender values are missing.",
        "Some transaction amounts are missing.",
        "Some channel values use inconsistent capitalisation.",
        "Some geographical values are inconsistent.",
        "A small number of transaction records have been duplicated.",
        "Some transaction amounts require investigation.",
        "Some text values contain leading or trailing spaces.",
        "A small number of province values use inconsistent naming."
    ]
})


# ============================================================
# 8. CREATE EXCEL WORKBOOK
# ============================================================

file_name = "Zanaco_Banking_Analytics_Raw_Data.xlsx"

with pd.ExcelWriter(
    file_name,
    engine="openpyxl"
) as writer:

    customers.to_excel(
        writer,
        sheet_name="Customers",
        index=False
    )

    accounts.to_excel(
        writer,
        sheet_name="Accounts",
        index=False
    )

    transactions.to_excel(
        writer,
        sheet_name="Transactions",
        index=False
    )

    loans.to_excel(
        writer,
        sheet_name="Loans",
        index=False
    )

    branches.to_excel(
        writer,
        sheet_name="Branches",
        index=False
    )

    data_quality_notes.to_excel(
        writer,
        sheet_name="DATA_QUALITY_NOTES",
        index=False
    )


# ============================================================
# 9. VERIFY THE DATA
# ============================================================

print("=" * 60)
print("ZANACO BANKING ANALYTICS DATASET")
print("=" * 60)

print(f"Customers:    {len(customers):,}")
print(f"Accounts:     {len(accounts):,}")
print(f"Transactions: {len(transactions):,}")
print(f"Loans:        {len(loans):,}")
print(f"Branches:     {len(branches):,}")

print("\nData preview:")
print("\nCUSTOMERS")
print(customers.head())

print("\nTRANSACTIONS")
print(transactions.head())

print("\nLOANS")
print(loans.head())

print("\n" + "=" * 60)
print("Dataset generation complete.")
print(f"File created: {file_name}")
print("=" * 60)

ZANACO BANKING ANALYTICS DATASET
Customers:    1,000
Accounts:     1,000
Transactions: 6,003
Loans:        450
Branches:     10

Data preview:

CUSTOMERS
  Customer_ID  Age  Gender    Province         City Branch_ID  \
0      C00001   64    Male      Lusaka       Lusaka      B003   
1      C00002   29    Male     Eastern      Chipata      B010   
2      C00003   33    Male    Southern  Livingstone      B007   
3      C00004   41  Female  Copperbelt        Kitwe      B005   
4      C00005   36  Female      Lusaka       Lusaka      B001   

  Customer_Segment  Join_Date  
0       Individual 2026-01-22  
1          Student 2025-02-19  
2              SME 2026-04-09  
3       Individual 2024-08-28  
4              SME 2024-10-22  

TRANSACTIONS
  Transaction_ID Customer_ID Transaction_Date Transaction_Type  \
0       T0000001      C00747       2026-05-03         Transfer   
1       T0000002      C00232       2026-06-11          Payment   
2       T0000003      C00007       2026-01-27      

In [ ]:
from google.colab import files

files.download("Zanaco_Banking_Analytics_Raw_Data.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>